In [3]:
import time
from concurrent.futures import ThreadPoolExecutor,as_completed,TimeoutError
def unstable_api(task_id:int):
    if task_id == 2:
        time.sleep(10)
    time.sleep(1)
    return f"任务 {task_id} 成功"
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(unstable_api,i) for i in range(5)]
    for future in as_completed(futures):
        try:
            result = future.result(timeout=3)
            print(f"✅ {result}")
        except TimeoutError:
            print(f"⏰ 任务超时！已放弃等待（主线程继续，子线程在后台自行结束）")

✅ 任务 0 成功
✅ 任务 1 成功
✅ 任务 3 成功
✅ 任务 4 成功
✅ 任务 2 成功


In [6]:
import time
from concurrent.futures import ThreadPoolExecutor
def embed_batch(batch:list[str]) -> int:
    time.sleep(1)
    return len(batch)
texts = [f"文本 {i}" for i in range(50)]
BATCH_SIZE = 10
batches = [texts[i:i + BATCH_SIZE] for i in range(0,len(texts),BATCH_SIZE)]
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(embed_batch,batches))
print(f"✅ 共 {len(texts)} 条文本，分 {len(batches)} 批并发完成，生成 {sum(results)} 个向量")

✅ 共 50 条文本，分 5 批并发完成，生成 50 个向量


In [16]:
import time,random
from concurrent.futures import ThreadPoolExecutor,as_completed
from tqdm import tqdm
def process_doc(doc_id:int):
    time.sleep(random.uniform(0.1,0.3))
    return f"Doc_{doc_id}"
with ThreadPoolExecutor(max_workers=20) as pool:
    futures = [pool.submit(process_doc,i) for i in range(100)]
    for future in tqdm(as_completed(futures),total=len(futures),desc="处理进度"):
        future.result()

处理进度: 100%|██████████| 100/100 [00:01<00:00, 85.87it/s]


In [ ]:
import time,threading,queue,random
task_queue = queue.Queue(maxsize=10)
def producer():
    for i in range(15):
        task_queue.put(f"URL_{i}")
        time.sleep(0.1)
    for _ in range(2):task_queue.put(None)
def consumer(name:str):
    while True:
        url = task_queue.get()
        if url is None:break
        time.sleep(0.5)
        task_queue.task_done()
for t in [threading.Thread(target=producer),
          threading.Thread(target=consumer,args=('LLM-1',)),
          threading.Thread(target=consumer,args=('LLM-2',))]:
        t.start()
task_queue.join()

In [ ]:
import time
import random
import threading
from concurrent.futures import ThreadPoolExecutor,as_completed
from dataclasses import dataclass
from typing import List
from tqdm import tqdm
#数据模型与统计器
@dataclass
class Chunk:
    doc_id:int
    text:str
    tokens:int = 0
class Metrics:
    def __init__(self):
        self._lock = threading.Lock()
        self.total_chunks = 0
        self.errors = 0
    def add_chunk(self):
        with self._lock:self.total_chunks += 1
    def add_error(self):
        with self._lock:self.errors += 1
#核心处理函数
def read_and_chunk(doc_id:int) -> List[Chunk]:
    time.sleep(random.uniform(0.05,0.15))
    return [Chunk(doc_id=doc_id,text=f"chunk_{i}",tokens=100)for i in range(3)]
def get_embeddings_batch(chunks:List[Chunk],metrics:Metrics) -> List[Chunk]:
    time.sleep(random.uniform(0.2,0.5))
    if random.random() < 0.05:
        metrics.add_error()
        raise Exception("API 500 Error")
    for c in chunks:
        metrics.add_chunk()
    return chunks
#主编排逻辑
def run_rag_pipeline(num_docs:int=50):
    metrics = Metrics()
    start_time = time.time()
    print(f"🚀 开始 RAG 数据预处理 (共 {num_docs} 个文档)...\n")
    all_chunks = []
    with ThreadPoolExecutor(max_workers=10) as pool:
        futures = [pool.submit(read_and_chunk,i) for i in range(num_docs)]
        for f in tqdm(as_completed(futures),total=num_docs,desc="1. 读取与分块"):
            all_chunks.extend(f.result())
        BATCH_SIZE = 10
        batches = [all_chunks[i:i+BATCH_SIZE] for i in range(0,len(all_chunks),BATCH_SIZE)]
        with ThreadPoolExecutor(max_workers=5) as pool:
            futures = [pool.submit(get_embeddings_batch,b,metrics) for b in batches]
            for f in tqdm(as_completed(futures),total=len(batches),desc="2. 生成 Embedding"):
                try:
                    f.result()
                except Exception:
                    pass
        print(f"\n✅ 完成！耗时: {time.time() - start_time:.2f}s")
        print(f"📊 成功处理 Chunk: {metrics.total_chunks} | 失败批次: {metrics.errors}")
if __name__ == '__main__':
    run_rag_pipeline(50)

🚀 开始 RAG 数据预处理 (共 50 个文档)...



2. 生成 Embedding: 100%|██████████| 15/15 [00:01<00:00, 11.59it/s]


✅ 完成！耗时: 1.91s
📊 成功处理 Chunk: 140 | 失败批次: 1


In [2]:
import asyncio
import time
def make_coffee_sync():
    print("☕ 开始做咖啡 (同步)...")
    time.sleep(2)
    print("☕ 咖啡做好了 (同步)")
async def make_coffee_async():
    print("☕ 开始做咖啡 (异步)...")
    await asyncio.sleep(2)
    print("☕ 咖啡做好了 (异步)")
start = time.time()
make_coffee_sync()
make_coffee_sync()
print(f"同步耗时: {time.time() - start:.2f}s\n")

async def main():
    start = time.time()
    await make_coffee_async()
    await make_coffee_async()
    print(f"串行异步耗时: {time.time() - start:.2f}s")
await main()

☕ 开始做咖啡 (同步)...
☕ 咖啡做好了 (同步)
☕ 开始做咖啡 (同步)...
☕ 咖啡做好了 (同步)
同步耗时: 4.00s

☕ 开始做咖啡 (异步)...
☕ 咖啡做好了 (异步)
☕ 开始做咖啡 (异步)...
☕ 咖啡做好了 (异步)
串行异步耗时: 4.02s
